[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishansurdi/SDK-Sovereign/blob/master/notebooks/03_train_auditor.ipynb)

# 03 — Train Auditor Adapter (GRPO)

**Purpose:** Fine-tune the Auditor LoRA adapter. Mirrors notebook 02 but with an auditor-specific reward function (allow-list correctness).

**Runtime:** Colab T4 · ~2h

> Run after notebook 02. Set `HF_TOKEN` + `WANDB_API_KEY` in Colab Secrets.

In [ ]:
# Cell 1 — Install + clone repo
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub datasets
!git clone https://github.com/ishansurdi/SDK-Sovereign.git sdk_sovereign_pkg 2>/dev/null || (cd sdk_sovereign_pkg && git pull)
!pip install -q -e sdk_sovereign_pkg/
import sys; sys.path.insert(0, 'sdk_sovereign_pkg')
print('✓ Ready')

In [ ]:
# Cell 2 — Config
HF_USER  = 'ishansurdi'
ENV_URL  = 'https://ishansurdi-sdk-sovereign.hf.space'
WANDB_PROJECT = 'sdk-sovereign'
WANDB_ENTITY = 'OpenEnvR2'

In [ ]:
# Cell 3 — Load model + agents (re-run if kernel is fresh)
from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from google.colab import userdata
from huggingface_hub import login
import wandb, os, json
from pathlib import Path

login(token=userdata.get('HF_TOKEN'))
wandb_key = userdata.get('WANDB_API_KEY')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    wandb.login(key=wandb_key)

model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print('✓ Model + agents loaded')

# Load auditor rollout data (written by notebook 02)
auditor_data = [json.loads(l) for l in Path('rollout_auditor.jsonl').read_text().splitlines()]
print(f'Auditor samples: {len(auditor_data)}')

In [ ]:
# Cell B — GRPO train Auditor adapter
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
from server.environment import SDKSovereignEnvironment

wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name='auditor-grpo-round1')

ds_auditor = Dataset.from_dict({'prompt': [x['prompt'] for x in auditor_data]})

def auditor_reward_fn(completions, **kwargs):
    rewards = []
    allowlist = set(SDKSovereignEnvironment().allowlist)
    for completion in completions:
        action = agents['auditor']._parse_action(completion)
        if action.action_type == 'approve':
            rewards.append(1.0)
        elif action.action_type == 'reject':
            rewards.append(0.5 if (action.rejection_reason or '').strip() else 0.1)
        elif action.action_type == 'give_hint':
            text = (action.hint_response or '').lower()
            hit = any(sdk.lower() in text for sdk in allowlist)
            rewards.append(0.8 if hit else -0.2)
        elif action.action_type == 'pass':
            rewards.append(-0.5)
        else:
            rewards.append(-0.2)
    return rewards

config = GRPOConfig(
    output_dir='checkpoints/auditor',
    num_generations=4,
    max_completion_length=256,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=2,
    report_to='wandb',
)

model.set_adapter('auditor_adapter')
for n, p in model.named_parameters():
    p.requires_grad = ('auditor_adapter' in n)
for n, p in model.named_parameters():
    if 'lead_adapter' in n:
        p.requires_grad = False

trainer = GRPOTrainer(
    model=model,
    reward_funcs=auditor_reward_fn,
    args=config,
    train_dataset=ds_auditor.select(range(min(60, len(ds_auditor)))),
    tokenizer=tokenizer,
 )
trainer.train()
wandb.finish()
print('✓ Auditor GRPO training complete')

In [ ]:
# Cell C — Save and push Auditor adapter to HF Hub
from huggingface_hub import HfApi

model.save_pretrained('checkpoints/auditor_adapter_v1', selected_adapters=['auditor_adapter'])
HfApi().upload_folder(
    folder_path='checkpoints/auditor_adapter_v1',
    repo_id=f'{HF_USER}/sdk-sovereign-auditor-adapter',
    repo_type='model',
    commit_message='Auditor LoRA adapter v1 (GRPO round 1)',
)
print(f'✓ Auditor adapter pushed to HF Hub: {HF_USER}/sdk-sovereign-auditor-adapter')